In [2]:
import pandas as pd
import numpy as np
import re

from scipy.sparse import hstack, csr_matrix

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from lightgbm import LGBMRegressor

In [24]:
from sklearn.decomposition import PCA

In [ ]:
train_df = pd.read_csv(
    "/content/train.csv"
)

train_df.shape

In [52]:
sample_df = train_df.sample(
    20000,
    random_state=42
).reset_index(drop=True)

sample_df.shape

(20000, 4)

In [53]:
X_train_text, X_valid_text, y_train, y_valid = train_test_split(
    sample_df["catalog_content"],
    sample_df["price"],
    test_size=0.2,
    random_state=42
)

In [54]:
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words="english"
)

X_train_tfidf = tfidf.fit_transform(
    X_train_text
)

X_valid_tfidf = tfidf.transform(
    X_valid_text
)

print(X_train_tfidf.shape)
print(X_valid_tfidf.shape)

(16000, 5000)
(4000, 5000)


In [55]:
import re
import numpy as np
import pandas as pd


def extract_quantity_features(text):
    text = str(text).lower()

    features = {}

    patterns = {
        "ounce_feature": r'(\d+(?:\.\d+)?)\s*(?:oz|ounce|ounces)\b',
        "pound_feature": r'(\d+(?:\.\d+)?)\s*(?:lb|lbs|pound|pounds)\b',
        "gram_feature": r'(\d+(?:\.\d+)?)\s*(?:g|gram|grams)\b',
        "kg_feature": r'(\d+(?:\.\d+)?)\s*(?:kg|kilogram|kilograms)\b',
        "ml_feature": r'(\d+(?:\.\d+)?)\s*(?:ml|milliliter|milliliters)\b',
        "liter_feature": r'(\d+(?:\.\d+)?)\s*(?:l|liter|liters)\b',

        "pack_feature": r'pack of\s*(\d+)|(\d+)[-\s]*pack\b',
        "count_feature": r'(\d+)\s*(?:count|ct)\b',
        "serving_feature": r'(\d+)\s*(?:servings?)\b',
        "bottle_feature": r'(\d+)\s*bottles?\b',
        "bag_feature": r'(\d+)\s*bags?\b',
        "case_feature": r'(\d+)\s*case\b',
        "dozen_feature": r'(\d+)\s*dozen\b'
    }

    for feature_name, pattern in patterns.items():
        matches = re.findall(pattern, text)

        numbers = []

        for match in matches:
            if isinstance(match, tuple):
                match = [m for m in match if m != ""]
                if len(match) > 0:
                    numbers.extend(match)
            else:
                numbers.append(match)

        numbers = [
            float(x)
            for x in numbers
            if x != "" and float(x) < 1000
        ]

        features[feature_name] = max(numbers) if numbers else 0

    return pd.Series(features)

In [56]:
quantity_features = train_df[
    "catalog_content"
].apply(extract_quantity_features)

quantity_features.head()

,ounce_feature,pound_feature,gram_feature,kg_feature,ml_feature,liter_feature,pack_feature,count_feature,serving_feature,bottle_feature,bag_feature,case_feature,dozen_feature
0,12.00,0.0,0.0,0.0,0.0,0.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0
1,8.00,0.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1.90,0.0,0.0,0.0,0.0,0.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0
3,11.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,12.70,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
quantity_features.describe()

In [57]:
for col in quantity_features.columns:
    print("=" * 100)
    print(col)

    suspicious_rows = train_df[
        quantity_features[col] > quantity_features[col].quantile(0.999)
    ][["catalog_content"]]

    print(suspicious_rows.head(10))

ounce_feature
                                        catalog_content
1122  Item Name: Lundberg Family Farms Eco-Farmed Sh...
1523  Item Name: Domino Sugar Dark Brown, 16-Ounce B...
2044  Item Name: Domino Sugar, Confectioners, 16-Oun...
2119  Item Name: Conchita Fancy Tomato Sauce, 8-Ounc...
3131  Item Name: Crystal Cayenne Hot Sauce with Garl...
3631  Item Name: Goya Foods Tomato Paste, 6 Ounce (P...
6070  Item Name: Rani Platinum White Basmati Rice Ex...
6107  Item Name: Mazola Corn Plus, 96-Ounce (Pack of...
8920  Item Name: Jack Link's Beef Jerky 5 Count Mult...
9739  Item Name: ENVIRO LOG Firelog 6 Pack, 412.8 OZ...
pound_feature
                                         catalog_content
683    Item Name: Country Brook Design - 1/2 Inch Wat...
4481   Item Name: Rubbermaid Commercial Products 2-Sh...
4735   Item Name: East West Furniture HLDU3-MAH-W 3 P...
5258   Item Name: Demi Doux Low Sugar Soda - Low Suga...
9330   Item Name: Kellogg's Pop Tarts Bars, Strawberr...
11329  Item Na

In [ ]:
quantity_features[
    quantity_features["pack_feature"] > 1000
].head(20)

In [61]:
quantity_train = quantity_features.loc[
    X_train_text.index
]

quantity_valid = quantity_features.loc[
    X_valid_text.index
]

print(quantity_train.shape)
print(quantity_valid.shape)

(16000, 13)
(4000, 13)


In [62]:
quantity_train_sparse = csr_matrix(
    quantity_train.values
)

quantity_valid_sparse = csr_matrix(
    quantity_valid.values
)

In [63]:
import requests
from PIL import Image
from io import BytesIO
from tqdm import tqdm
import torch
import timm

from torchvision import transforms

In [64]:
model = timm.create_model(
    "resnet50",
    pretrained=True,
    num_classes=0
)

model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (act1): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act1): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (drop_block): Identity()
      (act2): ReLU(inplace=True)
      (aa): Identity()
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
     

In [65]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

In [66]:
def get_image_embedding(url):
    try:
        response = requests.get(
            url,
            timeout=10
        )

        image = Image.open(
            BytesIO(response.content)
        ).convert("RGB")

        image = transform(image)

        image = image.unsqueeze(0)

        with torch.no_grad():
            embedding = model(image)

        return embedding.squeeze().numpy()

    except:
        return np.zeros(2048)

In [67]:
X_train_images = sample_df.loc[
    X_train_text.index,
    "image_link"
]

X_valid_images = sample_df.loc[
    X_valid_text.index,
    "image_link"
]

In [ ]:
train_embeddings = []

for url in tqdm(X_train_images):
    train_embeddings.append(
        get_image_embedding(url)
    )

train_embeddings = np.array(
    train_embeddings
)

 68%|██████▊   | 10920/16000 [56:47<25:45,  3.29it/s]

In [17]:
valid_embeddings = []

for url in tqdm(X_valid_images):
    valid_embeddings.append(
        get_image_embedding(url)
    )

valid_embeddings = np.array(
    valid_embeddings
)

100%|██████████| 1000/1000 [04:10<00:00,  3.99it/s]


In [ ]:
print(train_embeddings.shape)
print(valid_embeddings.shape)

In [25]:
pca = PCA(n_components=100)

train_embeddings_pca = pca.fit_transform(
    train_embeddings
)

valid_embeddings_pca = pca.transform(
    valid_embeddings
)

print(train_embeddings_pca.shape)
print(valid_embeddings_pca.shape)

train_image_embeddings = csr_matrix(
    train_embeddings_pca
)

valid_image_embeddings = csr_matrix(
    valid_embeddings_pca
)

(4000, 100)
(1000, 100)


In [26]:
X_train_final = hstack([
    X_train_tfidf,
    quantity_train_sparse,
    train_image_embeddings
])

X_valid_final = hstack([
    X_valid_tfidf,
    quantity_valid_sparse,
    valid_image_embeddings
])

print(X_train_final.shape)
print(X_valid_final.shape)

(4000, 5105)
(1000, 5105)


In [ ]:
multimodal_model = LGBMRegressor(
    n_estimators=800,
    learning_rate=0.03,
    num_leaves=50,
    max_depth=10,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

multimodal_model.fit(
    X_train_final,
    y_train
)

In [ ]:
multimodal_predictions = multimodal_model.predict(
    X_valid_final
)

In [ ]:
mae = mean_absolute_error(
    y_valid,
    multimodal_predictions
)

rmse = np.sqrt(
    mean_squared_error(
        y_valid,
        multimodal_predictions
    )
)

r2 = r2_score(
    y_valid,
    multimodal_predictions
)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2 Score:", r2)